[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/25-vision-transformers.ipynb)

# Module 25: Vision Transformers and Contrastive Learning

Companion notebook to [Module 25](https://transformers-and-math.drmhse.com/modules/25-vision-transformers) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)

## Part 1 — Convolutions (section 25.1)

In [ ]:
img = torch.zeros(1, 1, 20, 20); img[..., 6:14, 6:14] = 1          # a white square
filters = {
    'vertical edge': torch.tensor([[-1., 0, 1]] * 3),
    'horizontal edge': torch.tensor([[-1., 0, 1]] * 3).T,
    'blur': torch.full((3, 3), 1 / 9),
}
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(img[0, 0], cmap='gray'); axes[0].set_title('input')
for ax, (name, k) in zip(axes[1:], filters.items()):
    ax.imshow(F.conv2d(img, k[None, None])[0, 0], cmap='RdBu'); ax.set_title(name)
plt.show()

## Part 2 — An image as a sequence of patches (section 25.2)

In [ ]:
image = torch.randn(1, 3, 224, 224)
P = 16
patches = F.unfold(image, kernel_size=P, stride=P).transpose(1, 2)   # (1, 196, 768)
print('patches:', tuple(patches.shape))
embed = nn.Linear(3 * P * P, 192)
cls = nn.Parameter(torch.zeros(1, 1, 192))
pos = nn.Parameter(torch.zeros(1, 197, 192))
tokens = torch.cat([cls, embed(patches)], 1) + pos
encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(192, 3, 768, batch_first=True), 2)
print('encoder output:', tuple(encoder(tokens).shape), '— the same blocks as for text')

## Part 3 — Contrastive training, CLIP-style (section 25.3)
Two encoders, one for "images" and one for "captions", trained so each image is most similar to its own caption. The data is synthetic: each pair shares a hidden concept, seen through different random projections. With only 50 concepts, a batch of 128 contains repeats, which count as negatives here; that is why the loss levels off above zero.

In [ ]:
n_concepts, d_img, d_txt, d = 50, 64, 32, 16
proj_img, proj_txt = torch.randn(n_concepts, d_img), torch.randn(n_concepts, d_txt)

def pairs(B):
    c = torch.randint(0, n_concepts, (B,))
    return proj_img[c] + 0.3 * torch.randn(B, d_img), proj_txt[c] + 0.3 * torch.randn(B, d_txt), c

img_enc = nn.Sequential(nn.Linear(d_img, 64), nn.ReLU(), nn.Linear(64, d))
txt_enc = nn.Sequential(nn.Linear(d_txt, 64), nn.ReLU(), nn.Linear(64, d))
log_temp = nn.Parameter(torch.tensor(2.0))
opt = torch.optim.Adam([*img_enc.parameters(), *txt_enc.parameters(), log_temp], lr=3e-3)

for step in range(800):
    xi, xt, _ = pairs(128)
    zi, zt = F.normalize(img_enc(xi), dim=1), F.normalize(txt_enc(xt), dim=1)
    logits = zi @ zt.T * log_temp.exp()
    labels = torch.arange(128)
    loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        print(f'step {step:3}  InfoNCE loss {loss.item():.3f}   (chance level log 128 = {torch.log(torch.tensor(128.)):.3f})')

## Part 4 — Zero-shot classification (section 25.4)
Classify new images by comparing them with one "caption" per concept, with no classifier trained.

In [ ]:
with torch.no_grad():
    class_text = F.normalize(txt_enc(proj_txt), dim=1)            # one clean caption per concept
    xi, _, c = pairs(1000)
    pred = (F.normalize(img_enc(xi), dim=1) @ class_text.T).argmax(1)
print(f'zero-shot accuracy over {n_concepts} concepts: {(pred == c).float().mean():.1%}  (chance {1 / n_concepts:.0%})')

## Your turn
1. Replace the softmax loss with SigLIP's sigmoid loss: binary cross-entropy on every entry of `logits` (plus a learned bias), with 1 on the diagonal.
2. How many patches does a 336 × 336 image give with 14 × 14 patches? Check with `F.unfold`.